In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV

In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
sample_submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')

# Quick look at training data
print("Training Data Shape:", train.shape)
print("\nFirst 5 rows:")
train.head()

Training Data Shape: (594194, 21)

First 5 rows:


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
print("Data Types and Missing Values:")
print(train.info())

Data Types and Missing Values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  ob

In [5]:
print("Basic Statistics:")
print(train.describe())

Basic Statistics:
                  id  SeniorCitizen         tenure  MonthlyCharges  \
count  594194.000000  594194.000000  594194.000000   594194.000000   
mean   297096.500000       0.114102      36.577258       65.866223   
std    171529.177262       0.317936      25.061922       31.067444   
min         0.000000       0.000000       1.000000       18.250000   
25%    148548.250000       0.000000      12.000000       29.900000   
50%    297096.500000       0.000000      35.000000       74.100000   
75%    445644.750000       0.000000      62.000000       90.800000   
max    594193.000000       1.000000      72.000000      118.750000   

        TotalCharges  
count  594194.000000  
mean     2494.377057  
std      2353.916710  
min        18.800000  
25%       639.650000  
50%      1433.650000  
75%      4263.800000  
max      8684.800000  


In [6]:
print("Target Distribution:")
print(train['Churn'].value_counts(normalize=True))

Target Distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64


In [7]:
# Save ids and target
train_id = train['id']
test_id = test['id']
y = (train['Churn'] == 'Yes').astype(int)

# Drop id and target from features
train = train.drop(['id', 'Churn'], axis=1)
test = test.drop('id', axis=1)

# --- Feature Engineering (same as before) ---
def engineer_features(df):
    df = df.copy()
    # Tenure groups
    df['tenure_group'] = pd.cut(df['tenure'], bins=[0,6,12,24,48,100], labels=['0-6','6-12','12-24','24-48','48+'])
    # Avg monthly
    df['avg_monthly'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    # Binary flags for services
    service_cols = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                    'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    for col in service_cols:
        df[col + '_bin'] = (df[col] == 'Yes').astype(int)
    df['service_count'] = df[[col+'_bin' for col in service_cols]].sum(axis=1)
    df['has_internet'] = (df['InternetService'] != 'No').astype(int)
    # Polynomials
    df['tenure_sq'] = df['tenure'] ** 2
    df['monthly_sq'] = df['MonthlyCharges'] ** 2
    df['tenure_x_monthly'] = df['tenure'] * df['MonthlyCharges']
    return df

train_fe = engineer_features(train)
test_fe = engineer_features(test)

# --- Encode all categoricals (including new ones) ---
# Combine for consistent encoding
combined = pd.concat([train_fe, test_fe], axis=0, ignore_index=True)

# Identify object columns
cat_cols = combined.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns to encode:", cat_cols)

# Label encode each
for col in cat_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

# Split back
train_enc = combined.iloc[:len(train_fe)].copy()
test_enc = combined.iloc[len(train_fe):].copy()

# Add target back to train_enc (optional)
train_enc['Churn'] = y.values

# Verify all columns are numeric
print("\nData types after encoding:")
print(train_enc.dtypes.value_counts())

Categorical columns to encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Data types after encoding:
int64       29
float64      5
category     1
Name: count, dtype: int64


In [8]:
# Combine train_fe and test_fe
combined = pd.concat([train_fe, test_fe], axis=0, ignore_index=True)

# Encode ALL categorical columns in combined (both object and category dtypes)
# First, handle object columns with LabelEncoder

cat_cols = combined.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

# Convert any pandas 'category' columns to integer codes (e.g., tenure_group)
cat_dtype_cols = combined.select_dtypes(include=['category']).columns.tolist()
for col in cat_dtype_cols:
    combined[col] = combined[col].cat.codes

#  Now split back into train_enc and test_enc
train_enc = combined.iloc[:len(train_fe)].copy()
test_enc = combined.iloc[len(train_fe):].copy()

# Add target back to train_enc (using y from original train)
# (Make sure y is defined; if not, reload: original_train = pd.read_csv(...); y = (original_train['Churn'] == 'Yes').astype(int))
train_enc['Churn'] = y.values

# Verify all columns are numeric
print("Data types after encoding:")
print(train_enc.dtypes.value_counts())
print("\nAny missing values?", train_enc.isnull().any().any())

Data types after encoding:
int64      29
float64     5
int8        1
Name: count, dtype: int64

Any missing values? False


In [9]:
feature_cols = [col for col in train_enc.columns if col != 'Churn']
X = train_enc[feature_cols].copy()
print("X shape:", X.shape)
y = train_enc['Churn']
print("y shape:", y.shape)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X shape: (594194, 34)
y shape: (594194,)


In [10]:
# Assuming X (features) and y are already defined from your previous steps
# Make a copy to avoid modifying original
X_interact = X.copy()

# Contract is already encoded as 0,1,2 (likely 0=month-to-month, 1=one year, 2=two year)
# Create interactions with top continuous features
X_interact['Contract_x_tenure'] = X_interact['Contract'] * X_interact['tenure']
X_interact['Contract_x_MonthlyCharges'] = X_interact['Contract'] * X_interact['MonthlyCharges']
X_interact['Contract_x_TotalCharges'] = X_interact['Contract'] * X_interact['TotalCharges']

# Also with important binary features (OnlineSecurity, TechSupport, etc.)
X_interact['Contract_x_OnlineSecurity'] = X_interact['Contract'] * X_interact['OnlineSecurity']
X_interact['Contract_x_TechSupport'] = X_interact['Contract'] * X_interact['TechSupport']

# Optional: interactions among top continuous features
X_interact['tenure_x_MonthlyCharges'] = X_interact['tenure'] * X_interact['MonthlyCharges']  # already have tenure_x_monthly? might be duplicate

print("New features added. Shape:", X_interact.shape)

New features added. Shape: (594194, 40)


In [11]:
test_interact = test_enc.copy()

# Add interaction features (match what you did for X_interact)
test_interact['Contract_x_tenure'] = test_interact['Contract'] * test_interact['tenure']
test_interact['Contract_x_MonthlyCharges'] = test_interact['Contract'] * test_interact['MonthlyCharges']
test_interact['Contract_x_TotalCharges'] = test_interact['Contract'] * test_interact['TotalCharges']
test_interact['Contract_x_OnlineSecurity'] = test_interact['Contract'] * test_interact['OnlineSecurity']
test_interact['Contract_x_TechSupport'] = test_interact['Contract'] * test_interact['TechSupport']
test_interact['tenure_x_MonthlyCharges'] = test_interact['tenure'] * test_interact['MonthlyCharges']

# Verify columns now match
print("X_interact columns:", X_interact.columns.tolist())
print("\ntest_interact columns:", test_interact.columns.tolist())
assert all(X_interact.columns == test_interact.columns), "Columns still don't match!"

X_interact columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'tenure_group', 'avg_monthly', 'PhoneService_bin', 'MultipleLines_bin', 'OnlineSecurity_bin', 'OnlineBackup_bin', 'DeviceProtection_bin', 'TechSupport_bin', 'StreamingTV_bin', 'StreamingMovies_bin', 'service_count', 'has_internet', 'tenure_sq', 'monthly_sq', 'tenure_x_monthly', 'Contract_x_tenure', 'Contract_x_MonthlyCharges', 'Contract_x_TotalCharges', 'Contract_x_OnlineSecurity', 'Contract_x_TechSupport', 'tenure_x_MonthlyCharges']

test_interact columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',

In [13]:
param_dist = {
    'n_estimators': [200, 300, 400, 500, 600],
    'max_depth': [3, 5, 7, 9, -1],        # -1 means no limit
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1],
    'reg_lambda': [0.5, 1, 1.5, 2],
    'min_child_samples': [20, 30, 50, 100]   # LightGBM's equivalent of min_child_weight
}

In [14]:
lgb_model = LGBMClassifier(random_state=42, verbose=-1, force_col_wise=True)  # force_col_wise can speed up

random_search = RandomizedSearchCV(
    lgb_model,
    param_distributions=param_dist,
    n_iter=50,                # Number of parameter combinations to try
    cv=skf,                   # Your StratifiedKFold object
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_interact, y)

print("Best CV AUC:", random_search.best_score_)
print("Best parameters:", random_search.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best CV AUC: 0.9163886852510231
Best parameters: {'subsample': 0.6, 'reg_lambda': 1.5, 'reg_alpha': 0.1, 'n_estimators': 400, 'min_child_samples': 100, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.6}


In [15]:
best_lgb = random_search.best_estimator_
# Alternatively: best_lgb = LGBMClassifier(**random_search.best_params_, random_state=42)
best_lgb.fit(X_interact, y)

# Ensure test_interact has the same columns as X_interact (they should, but double-check)
assert all(X_interact.columns == test_interact.columns), "Column mismatch!"

lgb_preds = best_lgb.predict_proba(test_interact)[:, 1]

In [16]:
# Load original test.csv to get ids
test_original = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

submission_lgb = pd.DataFrame({
    'id': test_original['id'],
    'Churn': lgb_preds
})

submission_lgb.to_csv('lgb_submission.csv', index=False)
print("LightGBM submission saved! Preview:")
print(submission_lgb.head())

LightGBM submission saved! Preview:
       id     Churn
0  594194  0.057224
1  594195  0.000477
2  594196  0.117122
3  594197  0.003602
4  594198  0.502148
